In [1]:
import os

# Limit per-process BLAS/OpenMP threads; required for efficient process-level parallelism.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import time
import pennylane as qml
from pennylane import numpy as np
from pennylane import qchem
import re
from scipy.optimize import minimize
import warnings
from multiprocessing import get_context, cpu_count

warnings.filterwarnings("ignore")

r = 1.88973  # 1 Angstrom in Bohr

# Setup constants
symbols = ["H", "H", "H", "H"]
active_electrons = 4
active_orbitals = 4
charge = 0


def linear_h4_geometry(spacing_angstrom):
    s = float(spacing_angstrom) * r
    return np.array(
        [
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0 * s],
            [0.0, 0.0, 2.0 * s],
            [0.0, 0.0, 3.0 * s],
        ],
        requires_grad=False,
    )


# Default geometry corresponds to 3.0 Angstrom spacing.
geometry = linear_h4_geometry(3.0)


def ags_exact(symbols, geometry, active_electrons, active_orbitals, charge, adapt_it, shots=None):
    H, qubits = qml.qchem.molecular_hamiltonian(
        symbols,
        geometry,
        basis="sto-3g",
        method="pyscf",
        active_electrons=active_electrons,
        active_orbitals=active_orbitals,
        charge=charge,
    )
    hf_state = qchem.hf_state(active_electrons, qubits)

    ash_excitation = []
    gr_energies = []

    dev = qml.device("lightning.qubit", wires=qubits)

    @qml.qnode(dev)
    def commutator_0(H, w, k):
        qml.BasisState(k, wires=range(qubits))
        res = qml.commutator(H, w)
        return qml.expval(res)

    @qml.qnode(dev)
    def commutator_1(H, w, k):
        qml.StatePrep(k, wires=range(qubits))
        res = qml.commutator(H, w)
        return qml.expval(res)

    @qml.qnode(dev)
    def ash(params, ash_excitation, hf_state, H):
        [qml.PauliX(i) for i in np.nonzero(hf_state)[0]]
        for i, excitation in enumerate(ash_excitation):
            if len(ash_excitation[i]) == 4:
                qml.FermionicDoubleExcitation(
                    weight=params[i],
                    wires1=list(range(ash_excitation[i][0], ash_excitation[i][1] + 1)),
                    wires2=list(range(ash_excitation[i][2], ash_excitation[i][3] + 1)),
                )
            elif len(ash_excitation[i]) == 2:
                qml.FermionicSingleExcitation(
                    weight=params[i],
                    wires=list(range(ash_excitation[i][0], ash_excitation[i][1] + 1)),
                )
        return qml.expval(H)

    dev1 = qml.device("lightning.qubit", wires=qubits)

    @qml.qnode(dev1)
    def new_state(hf_state, ash_excitation, params):
        [qml.PauliX(i) for i in np.nonzero(hf_state)[0]]
        for i, excitations in enumerate(ash_excitation):
            if len(ash_excitation[i]) == 4:
                qml.FermionicDoubleExcitation(
                    weight=params[i],
                    wires1=list(range(ash_excitation[i][0], ash_excitation[i][1] + 1)),
                    wires2=list(range(ash_excitation[i][2], ash_excitation[i][3] + 1)),
                )
            elif len(ash_excitation[i]) == 2:
                qml.FermionicSingleExcitation(
                    weight=params[i],
                    wires=list(range(ash_excitation[i][0], ash_excitation[i][1] + 1)),
                )
        return qml.state()

    def cost(params):
        return ash(params, ash_excitation, hf_state, H)

    singles, doubles = qml.qchem.excitations(active_electrons, qubits)
    op1 = [qml.fermi.FermiWord({(0, x[0]): "+", (1, x[1]): "-"}) for x in singles]
    op2 = [
        qml.fermi.FermiWord({(0, x[0]): "+", (1, x[1]): "+", (2, x[2]): "-", (3, x[3]): "-"})
        for x in doubles
    ]
    operator_pool = op1 + op2

    states = [hf_state]
    params = np.zeros(len(ash_excitation), requires_grad=True)

    for _ in range(adapt_it):
        max_value = float("-inf")
        max_operator = None
        k = states[-1] if states else hf_state

        for op in operator_pool:
            w = qml.fermi.jordan_wigner(op)
            if np.array_equal(k, hf_state):
                current_value = abs(2 * commutator_0(H, w, k))
            else:
                current_value = abs(2 * commutator_1(H, w, k))

            if current_value > max_value:
                max_value = current_value
                max_operator = op

        indices_str = re.findall(r"\d+", str(max_operator))
        excitations = [int(index) for index in indices_str]
        ash_excitation.append(excitations)

        params = np.append(params, 0.0)
        result = minimize(
            cost,
            params,
            method="BFGS",
            tol=1e-12,
            options={"disp": False, "maxiter": int(1e8)},
        )

        gr_energies.append(result.fun)
        params = result.x

        ostate = new_state(hf_state, ash_excitation, params)
        states.append(ostate)

    return params, ash_excitation, gr_energies


def _run_geometry_worker(task):
    spacing_angstrom, adapt_it = task
    t0 = time.perf_counter()
    geom = linear_h4_geometry(spacing_angstrom)
    _, ash_excitation, gr_energies = ags_exact(
        symbols,
        geom,
        active_electrons,
        active_orbitals,
        charge,
        adapt_it=adapt_it,
    )
    return {
        "spacing_angstrom": float(spacing_angstrom),
        "final_energy": float(gr_energies[-1]),
        "num_excitations": int(len(ash_excitation)),
        "energies": [float(x) for x in gr_energies],
        "elapsed_s": float(time.perf_counter() - t0),
    }


In [ ]:
spacings_angstrom = [2.0, 3.0, 4.0]
adapt_it = 3

tasks = [(spacing, adapt_it) for spacing in spacings_angstrom]
# Keep worker count conservative for PySCF/PennyLane workloads.
num_workers = min(len(tasks), 2)

print(f"Running {len(tasks)} geometries with {num_workers} processes")

results = []
with get_context("fork").Pool(processes=num_workers, maxtasksperchild=1) as pool:
    for i, item in enumerate(pool.imap_unordered(_run_geometry_worker, tasks, chunksize=1), start=1):
        print(
            f"done {i}/{len(tasks)} | spacing={item['spacing_angstrom']:.1f}A | "
            f"E={item['final_energy']:.8f} Ha | t={item['elapsed_s']:.1f}s"
        )
        results.append(item)

results = sorted(results, key=lambda item: item["spacing_angstrom"])
print("\nSummary")
for item in results:
    print(f"spacing={item['spacing_angstrom']:.1f}A")
    print("ADAPT energies (Ha):", item["energies"])
    print("Final ADAPT energy (Ha):", item["final_energy"])
    print("num_excitations:", item["num_excitations"])
    print("elapsed_s:", item["elapsed_s"])
    print("-")


The final ground state energy is -1.726841684639911
ADAPT excitations: [[2, 3, 6, 7], [0, 3, 5, 6], [0, 1, 4, 5]]
